# 10 — Iacobus L1 vs L2 spread signal

Source: Discord user "iacobus" (untrusted) — claim: when L1 spread is wide vs L2, mid mean-reverts. Tested on R3 days 0/1/2 for `HYDROGEL_PACK` and `VELVETFRUIT_EXTRACT`.

Companion script: `scripts/eda_10_iacobus.py`. Findings doc: `docs/round_3/research/10_iacobus_l1_l2_signal.md`.

## Imports & config

Standard scientific stack plus inline plotting. Pin paths, products, and forward-return horizons up top so the rest of the notebook is parameter-free.

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "/Users/bensinek/Documents/Coding/Prosperity4/data/round_3"
PLOT_DIR = "/Users/bensinek/Documents/Coding/Prosperity4/docs/round_3/research/plots"
PRODUCTS = ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]
HORIZONS = [1, 5, 20]

## Load order book

Pull all 3 R3 days for a given product, concatenate, and tag each row with its day so we can compute forward returns without leaking across day boundaries. CSVs are semicolon-separated.

In [ ]:
def load_product(product):
    frames = []
    for d in [0, 1, 2]:
        df = pd.read_csv(f"{DATA_DIR}/prices_round_3_day_{d}.csv", sep=";")
        df = df[df["product"] == product].copy()
        df = df.sort_values("timestamp").reset_index(drop=True)
        df["day"] = d
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

## Analyze: L1/L2 spreads, distribution, buckets, correlations, plots

Build `L1_spread`, `L2_spread`, and `diff = L1 - L2` (negative = L1 wider than L2). Compute forward mid returns at h ∈ {1, 5, 20} ticks within day. Then summarize: distribution moments + quantiles, conditional means in extreme/middle buckets with t-stats, linear correlation of `diff` against forward returns, and two diagnostic plots (diff histogram + mean ret_5 by diff decile).

In [ ]:
def analyze(product):
    df = load_product(product)
    df["L1_spread"] = df["ask_price_1"] - df["bid_price_1"]
    df["L2_spread"] = df["ask_price_2"] - df["bid_price_2"]
    df["L1_minus_L2"] = df["L1_spread"] - df["L2_spread"]  # negative = L1 wider than L2
    # mid returns by horizon, computed within day
    out = {}
    for h in HORIZONS:
        df[f"ret_{h}"] = df.groupby("day")["mid_price"].shift(-h) - df["mid_price"]
    valid = df.dropna(subset=[f"ret_{h}" for h in HORIZONS]).copy()

    # Distribution stats
    diff = valid["L1_minus_L2"]
    out["n"] = len(valid)
    out["L1_mean"] = valid["L1_spread"].mean()
    out["L2_mean"] = valid["L2_spread"].mean()
    out["diff_mean"] = diff.mean()
    out["diff_std"] = diff.std()
    out["diff_quantiles"] = diff.quantile([0.01, 0.05, 0.5, 0.95, 0.99]).to_dict()

    # Bucket by diff sign and extremes
    q_lo = diff.quantile(0.05)
    q_hi = diff.quantile(0.95)
    extreme_neg = valid[diff <= q_lo]  # L1 much wider than L2 -> hypothesis: revert (mid moves toward fair)
    extreme_pos = valid[diff >= q_hi]  # L1 much tighter than L2

    bucket_stats = {}
    for name, sub in [("extreme_neg(L1>>L2)", extreme_neg),
                      ("extreme_pos(L1<<L2)", extreme_pos),
                      ("middle", valid[(diff > q_lo) & (diff < q_hi)])]:
        row = {"n": len(sub)}
        for h in HORIZONS:
            r = sub[f"ret_{h}"]
            mean = r.mean()
            se = r.std(ddof=1) / np.sqrt(len(r)) if len(r) > 1 else np.nan
            t = mean / se if se and se > 0 else np.nan
            row[f"ret_{h}_mean"] = mean
            row[f"ret_{h}_t"] = t
        bucket_stats[name] = row
    out["buckets"] = bucket_stats

    # Correlation: signed diff vs forward returns (linear edge test)
    corrs = {}
    for h in HORIZONS:
        c = valid[["L1_minus_L2", f"ret_{h}"]].corr().iloc[0, 1]
        # t for correlation
        n = len(valid)
        t = c * np.sqrt((n - 2) / max(1e-12, 1 - c * c))
        corrs[h] = (c, t)
    out["corrs"] = corrs

    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(diff, bins=80, color="steelblue", edgecolor="none")
    axes[0].set_title(f"{product}: L1_spread - L2_spread")
    axes[0].set_xlabel("L1 - L2 (ticks)")
    axes[0].set_ylabel("count")
    # mean forward return by diff bucket
    valid["bucket"] = pd.qcut(valid["L1_minus_L2"], q=10, duplicates="drop")
    grp = valid.groupby("bucket", observed=True)["ret_5"].mean()
    axes[1].bar(range(len(grp)), grp.values, color="darkorange")
    axes[1].set_title(f"{product}: mean ret_5 by L1-L2 decile")
    axes[1].set_xlabel("L1-L2 decile (low=L1 wider)")
    axes[1].set_ylabel("mean 5-tick mid return")
    axes[1].axhline(0, color="k", lw=0.5)
    plt.tight_layout()
    fname = f"{PLOT_DIR}/10_{product.lower()}_l1l2.png"
    plt.savefig(fname, dpi=110)
    plt.close()
    out["plot"] = fname
    return out

## Pretty-print helper

Format the per-product result dict into the same console layout used by the standalone script — distribution moments, correlations with t-stats, and bucket breakdowns at each horizon.

In [ ]:
def fmt(out, product):
    print(f"\n=== {product} ===")
    print(f"n={out['n']}  L1_mean={out['L1_mean']:.3f}  L2_mean={out['L2_mean']:.3f}")
    print(f"diff mean={out['diff_mean']:.3f} std={out['diff_std']:.3f}")
    print(f"diff quantiles: {out['diff_quantiles']}")
    print("Correlations (L1-L2 vs forward ret):")
    for h, (c, t) in out["corrs"].items():
        print(f"  h={h}: r={c:+.5f}  t={t:+.2f}")
    print("Bucket stats:")
    for name, row in out["buckets"].items():
        s = f"  {name}: n={row['n']}"
        for h in HORIZONS:
            s += f"  ret_{h}={row[f'ret_{h}_mean']:+.4f} (t={row[f'ret_{h}_t']:+.2f})"
        print(s)
    print(f"plot: {out['plot']}")

## Per-product run

Loop both products and print the full breakdown. Plots are saved to `docs/round_3/research/plots/`.

In [ ]:
for p in PRODUCTS:
    fmt(analyze(p), p)

## Summary + Key Findings

**Setup**
- L1_spread = ask1 − bid1; L2_spread = ask2 − bid2; `diff = L1 − L2`.
- Forward mid returns at h ∈ {1, 5, 20} ticks, within-day (no day-boundary leakage).
- n = 29,940 ticks/product (3 days × ~10k).

**Distribution**
- HYDROGEL_PACK: L1 mean 15.7, L2 mean 20.9, `diff` mean −5.18, std 1.06. ~88% of ticks have diff = −5; tails thin and sparse (only −12, −11, −10, −9, −6, −4 observed).
- VELVETFRUIT_EXTRACT: L1 mean 4.99, L2 mean 6.85, `diff` mean −2.26, std 0.68. ~77% at diff = −2; tail values −3, −4, −5.
- "L1 wider than L2" basically never happens; signal lives entirely in *how much narrower* L1 is.

**Linear correlation (diff vs forward ret)**
- HYDROGEL_PACK: r₁ = −0.053 (t=−9.2), r₅ = −0.026 (t=−4.5), r₂₀ = −0.016 (t=−2.9).
- VELVETFRUIT_EXTRACT: r₁ = −0.209 (t=−37.0), r₅ = −0.108 (t=−18.8), r₂₀ = −0.047 (t=−8.1).
- Negative slope = consistent with iacobus: L1 wider (more negative diff) → positive forward mid move. Strong on VFE, marginal on HYDROGEL.

**Conditional means — HYDROGEL_PACK** (mean ret_5 by `diff` bucket, n; 3-day pooled, stable across days)
- diff=−12 → r₅=+4.12 (n=253)
- diff=−11 → r₅=−2.71 (n=260)
- diff=−10 → r₅=+2.70 (n=267)
- diff=−9 → r₅=−3.55 (n=210)
- diff=−6 → r₅=−0.01 (n=1286)
- diff=−5 → r₅=−0.01 (n=26212)
- diff=−4 → r₅=+0.05 (n=1452)
- Tail buckets **alternate sign with parity** of diff → tick-grid / midpoint-rounding artifact (mid sits on a half-tick when L1 is odd), not a tradeable directional edge.

**Conditional means — VELVETFRUIT_EXTRACT** (stable across days)
- diff=−5 → r₅=+1.89 (n=174, t≈9)
- diff=−4 → r₅=+0.51 (n=585, t≈6)
- diff=−3 → r₅=−0.65 (n=441, t≈−6)
- diff=−2 → r₅=−0.07 (n=6930)
- Monotonic, sign-stable, statistically real. diff=−5 bucket gives +1.9-tick expected mid move over 5 ticks.

**Tradeability after costs**
- HYDROGEL_PACK: L1 spread 15.7 ticks → crossing cost ~7.8/side. Tail effect (max ~+4 ticks at diff=−12) well under spread cost; parity flip kills directional take. Even passive resting orders would only need a +4 quote skew in 0.9% of ticks — not worth a special branch.
- VELVETFRUIT_EXTRACT: L1 spread mean ~5 ticks → ~2.5/side. Effect at diff∈{−4,−5} is +0.5 to +1.9 ticks (n≈760, ~2.5% of ticks). Crossing burns the edge. **Passive** quote skew (lift bid / pull ask) when diff ≤ −4 could capture a fraction without paying the spread, but rare-event count (~250/day) limits PnL.

**Verdict — Tradeable? Marginal — N for HYDROGEL_PACK, weak-Y for VELVETFRUIT_EXTRACT (passive only).**
- HYDROGEL: tail "signal" is a midpoint-parity artifact, not directional. Skip.
- VFE: real, stable, monotonic edge in diff∈{−4,−5} tails (~2.5% of ticks, +0.5–1.9 tick expected move over 5t). Too thin to cross the spread; only worth wiring as a quote-skew bias inside an existing MM loop. Not a standalone strategy.